In [2]:
!pip install ultralytics
!pip install opencv-python-headless
!pip install pyttsx3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.2 MB/s eta 0:00:00


In [8]:
import cv2
from ultralytics import YOLO
import pyttsx3
import time

# Initialize TTS engine
engine = pyttsx3.init()
engine.setProperty('rate', 150)
engine.setProperty('volume', 1.0)

# Load YOLOv8 model
model = YOLO("/content/yolov8n.pt")  # path to your YOLOv8 model

# Open webcam
cap = cv2.VideoCapture(0)

# Cooldown to prevent repeated speech
last_spoken = {}
SPEAK_COOLDOWN = 3  # seconds

print("Starting YOLOv8 camera. Press 'q' to exit...")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # YOLOv8 inference
    results = model(frame)[0]  # first result
    objects_detected = []
    current_time = time.time()

    # Loop through detections
    for det in results.boxes:
        label = model.names[int(det.cls[0])]

        # Check cooldown
        if label not in last_spoken or current_time - last_spoken[label] > SPEAK_COOLDOWN:
            objects_detected.append(label)
            last_spoken[label] = current_time

        # Draw bounding box
        x1, y1, x2, y2 = map(int, det.xyxy[0])
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, label, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    # Speak detected objects
    if objects_detected:
        speech_text = " , ".join(objects_detected)
        engine.say(speech_text)
        engine.runAndWait()

    # Show camera frame
    cv2.imshow("YOLOv8 Object Detection for Blind Assistance", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Cleanup
cap.release()
cv2.destroyAllWindows()
engine.stop()


Starting YOLOv8 camera. Press 'q' to exit...


In [2]:
!pip install ultralytics opencv-python-headless pyttsx3
!sudo apt-get update && sudo apt-get install espeak-ng -y

import cv2
from ultralytics import YOLO
import pyttsx3
import gradio as gr
import time
import numpy as np
import os

# Initialize TTS engine
engine = pyttsx3.init()
engine.setProperty('rate', 150)
engine.setProperty('volume', 1.0)

# Load YOLOv8 model
model = YOLO("/content/yolov8n.pt")  # path to your YOLOv8 model

# Cooldown dictionary to prevent repeated speech
last_spoken = {}
SPEAK_COOLDOWN = 3  # seconds

# Placeholder calibration parameters (These would ideally be determined through a calibration process)
KNOWN_DISTANCE = 100  # Distance from camera to object during calibration (in cm)
KNOWN_WIDTH = 50  # Real-world width of the object used for calibration (e.g., average person shoulder width in cm)
FOCAL_LENGTH = 700  # Placeholder for focal length (This needs to be calibrated)
AUDIO_GAP = 2 # seconds gap between speaking each object


# Function to calculate distance to camera
def distance_to_camera(known_width, focal_length, pixel_width):
    """
    Calculates the distance of an object from the camera using triangle similarity.
    """
    # Add a small value to pixel_width to avoid division by zero,
    # although the check `if pixel_width > 0` should prevent this.
    # This is an extra precaution.
    return (known_width * focal_length) / (pixel_width + 1e-6)


# Function to process frame, detect objects, and generate speech audio
def detect_objects(frame):
    global last_spoken

    # Convert frame from RGB (Gradio) to BGR (OpenCV)
    frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

    # YOLOv8 inference
    results = model(frame_bgr)[0]

    objects_detected_info = [] # Store tuples of (label, distance)
    current_time = time.time()

    # Loop through detections
    for det in results.boxes:
        label = model.names[int(det.cls[0])]

        # Calculate bounding box width in pixels
        x1, y1, x2, y2 = map(int, det.xyxy[0])
        pixel_width = x2 - x1

        distance = None # Initialize distance to None
        # Estimate distance
        if pixel_width > 0: # Avoid division by zero
            # NOTE: For accurate distance estimation of various objects,
            # you would need known widths for each object class or a
            # more sophisticated depth estimation method.
            # Using a placeholder KNOWN_WIDTH here for demonstration.
            distance = distance_to_camera(KNOWN_WIDTH, FOCAL_LENGTH, pixel_width)


        # Check cooldown and add detection info
        if label not in last_spoken or current_time - last_spoken[label] > SPEAK_COOLDOWN:
            # Store label and distance for speaking later
            objects_detected_info.append((label, distance))
            last_spoken[label] = current_time # Update last spoken time for this label


        # Draw bounding box
        cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), (0, 255, 0), 2)
        # Add distance to the text displayed on the frame
        display_text = f"{label}: {distance:.2f} cm" if distance is not None else label
        cv2.putText(frame_bgr, display_text, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    # Generate speech audio based on collected information with gaps
    speech_text = ""
    audio_output = None

    if objects_detected_info:
        for i, (label, distance) in enumerate(objects_detected_info):
            speech_phrase = ""
            if distance is not None:
                speech_phrase = f"{label} at {distance:.2f} centimeters"
            else:
                speech_phrase = f"{label}"

            if speech_phrase:
                # Save the speech to a temporary file
                temp_audio_file = f"temp_speech_{i}.mp3" # Use unique filename for each object
                engine.save_to_file(speech_phrase, temp_audio_file)
                engine.runAndWait()
                # Read the audio file
                if os.path.exists(temp_audio_file):
                    # In a real-world scenario, you might want to play these audio files sequentially
                    # within the Gradio interface or a separate thread.
                    # For this example, we'll just save the last one or concatenate.
                    # Returning the last one for simplicity, or you could return a list of paths.
                    audio_output = temp_audio_file
                    # Add a delay after speaking each object
                    time.sleep(AUDIO_GAP)


    # Convert frame back to RGB for Gradio
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    # Return both the processed image and the audio file path
    # Note: Returning only the last audio file path in this simplified example.
    # A more robust solution would handle multiple audio outputs or concatenation.
    return frame_rgb, audio_output


# Gradio Interface
iface = gr.Interface(
    fn=detect_objects,
    inputs=gr.Image(streaming=True),
    outputs=["image", gr.Audio(label="Detected Objects Audio", autoplay=True)],
    live=True,
    title="YOLOv8 Object Detection for Blind Assistance",
    description="Uses webcam feed to detect objects and speaks them aloud."
)

iface.launch()

  Using cached ultralytics-8.3.228-py3-none-any.whl.metadata (37 kB)
  Using cached opencv_python_headless-4.12.0.88-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached pyttsx3-2.99-py3-none-any.whl.metadata (6.2 kB)
  Using cached opencv_python-4.12.0.88-cp37-abi3-win_amd64.whl.metadata (19 kB)
  Using cached torch-2.9.1-cp312-cp312-win_amd64.whl.metadata (30 kB)
  Using cached torchvision-0.24.1-cp312-cp312-win_amd64.whl.metadata (5.9 kB)
  Using cached polars-1.35.2-py3-none-any.whl.metadata (10 kB)
  Using cached ultralytics_thop-2.0.18-py3-none-any.whl.metadata (14 kB)
  Using cached numpy-2.2.6-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Using cached comtypes-1.4.13-py3-none-any.whl.metadata (7.2 kB)
  Using cached pypiwin32-223-py3-none-any.whl.metadata (236 bytes)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached polars_runtime_32-1.35.2-cp39-abi3-win_amd64.whl.metadata (1.5 kB)
INFO: pip is looking at multiple versions of contourpy to determine 

ERROR: Exception:
Traceback (most recent call last):
  File "C:\Users\thari\anaconda3\Lib\site-packages\pip\_vendor\urllib3\response.py", line 438, in _error_catcher
    yield
  File "C:\Users\thari\anaconda3\Lib\site-packages\pip\_vendor\urllib3\response.py", line 561, in read
    data = self._fp_read(amt) if not fp_closed else b""
           ^^^^^^^^^^^^^^^^^^
  File "C:\Users\thari\anaconda3\Lib\site-packages\pip\_vendor\urllib3\response.py", line 527, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
           ^^^^^^^^^^^^^^^^^^
  File "C:\Users\thari\anaconda3\Lib\site-packages\pip\_vendor\cachecontrol\filewrapper.py", line 102, in read
    self.__buf.write(data)
  File "C:\Users\thari\anaconda3\Lib\tempfile.py", line 499, in func_wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
OSError: [Errno 28] No space left on device

During handling of the above exception, another exception occurred:

Traceback (most recent call last)

ModuleNotFoundError: No module named 'cv2'

In [4]:
!sudo apt-get update
!sudo apt-get install espeak-ng

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:8 https://cli.github.com/packages stable InRelease [3,917 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 Packages [38.5 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://cli.github.com/packages stable/main amd64 Packages [3

In [11]:
import cv2
from ultralytics import YOLO
import pyttsx3
import gradio as gr
import time
import numpy as np
import os

# Initialize TTS engine (we'll still use it to generate the audio data)
engine = pyttsx3.init()
engine.setProperty('rate', 150)
engine.setProperty('volume', 1.0)

# Load YOLOv8 model
model = YOLO("/content/yolov8n.pt")  # path to your YOLOv8 model

# Cooldown dictionary to prevent repeated speech
last_spoken = {}
SPEAK_COOLDOWN = 3  # seconds

# Function to process frame, detect objects, and generate speech audio
def detect_objects(frame):
    global last_spoken

    # Convert frame from RGB (Gradio) to BGR (OpenCV)
    frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

    # YOLOv8 inference
    results = model(frame_bgr)[0]

    objects_detected = []
    current_time = time.time()

    for det in results.boxes:
        label = model.names[int(det.cls[0])]
        # Check cooldown
        if label not in last_spoken or current_time - last_spoken[label] > SPEAK_COOLDOWN:
            objects_detected.append(label)
            last_spoken[label] = current_time

        # Draw bounding box
        x1, y1, x2, y2 = map(int, det.xyxy[0])
        cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame_bgr, label, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    # Generate speech audio if objects were detected
    speech_text = ""
    audio_output = None
    if objects_detected:
        speech_text = " , ".join(objects_detected)
        # Save the speech to a temporary file
        temp_audio_file = "temp_speech.mp3"
        engine.save_to_file(speech_text, temp_audio_file)
        engine.runAndWait()
        # Read the audio file
        if os.path.exists(temp_audio_file):
            audio_output = temp_audio_file


    # Convert frame back to RGB for Gradio
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    # Return both the processed image and the audio file path
    return frame_rgb, audio_output

# Gradio Interface
iface = gr.Interface(
    fn=detect_objects,
    inputs=gr.Image(streaming=True),
    outputs=["image", gr.Audio(label="Detected Objects Audio", autoplay=True)],
    live=True,
    title="YOLOv8 Object Detection for Blind Assistance",
    description="Uses webcam feed to detect objects and speaks them aloud."
)

iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8361a0260906bd0453.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Task
Modify the code to detect only 'person' objects, estimate their distance, and speak the information.

## Filter for 'person' detections

### Subtask:
Modify the `detect_objects` function to only process detections where the detected object is a 'person'.


**Reasoning**:
Modify the `detect_objects` function to filter detections and only process 'person' objects as instructed.



In [12]:
# Function to process frame, detect objects, and generate speech audio
def detect_objects(frame):
    global last_spoken

    # Convert frame from RGB (Gradio) to BGR (OpenCV)
    frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

    # YOLOv8 inference
    results = model(frame_bgr)[0]

    objects_detected = []
    current_time = time.time()

    # Loop through detections
    for det in results.boxes:
        label = model.names[int(det.cls[0])]

        # Only process if the detected object is a 'person'
        if label == 'person':
            # Check cooldown
            if label not in last_spoken or current_time - last_spoken[label] > SPEAK_COOLDOWN:
                objects_detected.append(label)
                last_spoken[label] = current_time

            # Draw bounding box
            x1, y1, x2, y2 = map(int, det.xyxy[0])
            cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame_bgr, label, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    # Generate speech audio if objects were detected
    speech_text = ""
    audio_output = None
    if objects_detected:
        speech_text = " , ".join(objects_detected)
        # Save the speech to a temporary file
        temp_audio_file = "temp_speech.mp3"
        engine.save_to_file(speech_text, temp_audio_file)
        engine.runAndWait()
        # Read the audio file
        if os.path.exists(temp_audio_file):
            audio_output = temp_audio_file


    # Convert frame back to RGB for Gradio
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    # Return both the processed image and the audio file path
    return frame_rgb, audio_output

## Implement distance estimation

### Subtask:
Add code to calculate the distance of the detected person. This will likely involve a placeholder for calibration and a formula to estimate distance based on bounding box size.


**Reasoning**:
Add placeholder calibration variables and the distance calculation function, then integrate distance calculation into the detection loop for 'person' objects.



In [17]:
# Placeholder calibration parameters (These would ideally be determined through a calibration process)
KNOWN_DISTANCE = 100  # Distance from camera to object during calibration (in cm)
KNOWN_WIDTH = 50  # Real-world width of the object used for calibration (e.g., average person shoulder width in cm)
FOCAL_LENGTH = 700  # Placeholder for focal length (This needs to be calibrated)
AUDIO_GAP = 2 # seconds gap between speaking each object

# Function to calculate distance to camera
def distance_to_camera(known_width, focal_length, pixel_width):
    """
    Calculates the distance of an object from the camera using triangle similarity.
    """
    # Add a small value to pixel_width to avoid division by zero,
    # although the check `if pixel_width > 0` should prevent this.
    # This is an extra precaution.
    return (known_width * focal_length) / (pixel_width + 1e-6)


# Function to process frame, detect objects, and generate speech audio
def detect_objects(frame):
    global last_spoken

    # Convert frame from RGB (Gradio) to BGR (OpenCV)
    frame_bgr = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) # Corrected conversion to RGB for Gradio output

    # YOLOv8 inference
    results = model(frame_bgr)[0]

    objects_detected_info = [] # Store tuples of (label, distance)
    current_time = time.time()

    # Loop through detections
    for det in results.boxes:
        label = model.names[int(det.cls[0])]

        # Calculate bounding box width in pixels
        x1, y1, x2, y2 = map(int, det.xyxy[0])
        pixel_width = x2 - x1

        distance = None # Initialize distance to None
        # Estimate distance
        if pixel_width > 0: # Avoid division by zero
            # NOTE: For accurate distance estimation of various objects,
            # you would need known widths for each object class or a
            # more sophisticated depth estimation method.
            # Using a placeholder KNOWN_WIDTH here for demonstration.
            distance = distance_to_camera(KNOWN_WIDTH, FOCAL_LENGTH, pixel_width)


        # Check cooldown and add detection info
        if label not in last_spoken or current_time - last_spoken[label] > SPEAK_COOLDOWN:
            # Store label and distance for speaking later
            objects_detected_info.append((label, distance))
            last_spoken[label] = current_time # Update last spoken time for this label


        # Draw bounding box
        cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), (0, 255, 0), 2)
        # Add distance to the text displayed on the frame
        display_text = f"{label}: {distance:.2f} cm" if distance is not None else label
        cv2.putText(frame_bgr, display_text, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    # Generate speech audio based on collected information with gaps
    speech_text = ""
    audio_output = None

    if objects_detected_info:
        for i, (label, distance) in enumerate(objects_detected_info):
            speech_phrase = ""
            if distance is not None:
                speech_phrase = f"{label} at {distance:.2f} centimeters"
            else:
                speech_phrase = f"{label}"

            if speech_phrase:
                # Save the speech to a temporary file
                temp_audio_file = f"temp_speech_{i}.mp3" # Use unique filename for each object
                engine.save_to_file(speech_phrase, temp_audio_file)
                engine.runAndWait()
                # Read the audio file
                if os.path.exists(temp_audio_file):
                    # In a real-world scenario, you might want to play these audio files sequentially
                    # within the Gradio interface or a separate thread.
                    # For this example, we'll just save the last one or concatenate.
                    # Concatenating audio files with pyttsx3 is not straightforward.
                    # Returning the last one for simplicity, or you could return a list of paths.
                    audio_output = temp_audio_file
                    # Add a delay after speaking each object
                    time.sleep(AUDIO_GAP)


    # Convert frame back to RGB for Gradio
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    # Return both the processed image and the audio file path
    # Note: Returning only the last audio file path in this simplified example.
    # A more robust solution would handle multiple audio outputs or concatenation.
    return frame_rgb, audio_output


# Gradio Interface
iface = gr.Interface(
    fn=detect_objects,
    inputs=gr.Image(streaming=True),
    outputs=["image", gr.Audio(label="Detected Objects Audio", autoplay=True)],
    live=True,
    title="YOLOv8 Object Detection for Blind Assistance",
    description="Uses webcam feed to detect objects and speaks them aloud."
)

iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2324b555abc63b9f9d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Update speech output

### Subtask:
Modify the speech output to include information about the detected person and their estimated distance.


**Reasoning**:
Modify the speech generation logic within the `detect_objects` function to include the estimated distance when a 'person' is detected. This fulfills the current subtask instructions.



In [14]:
# Placeholder calibration parameters (These would ideally be determined through a calibration process)
KNOWN_DISTANCE = 100  # Distance from camera to object during calibration (in cm)
KNOWN_WIDTH = 50  # Real-world width of the object used for calibration (e.g., average person shoulder width in cm)
FOCAL_LENGTH = 700  # Placeholder for focal length (This needs to be calibrated)

# Function to calculate distance to camera
def distance_to_camera(known_width, focal_length, pixel_width):
    """
    Calculates the distance of an object from the camera using triangle similarity.
    """
    # Add a small value to pixel_width to avoid division by zero,
    # although the check `if pixel_width > 0` should prevent this.
    # This is an extra precaution.
    return (known_width * focal_length) / (pixel_width + 1e-6)


# Function to process frame, detect objects, and generate speech audio
def detect_objects(frame):
    global last_spoken

    # Convert frame from RGB (Gradio) to BGR (OpenCV)
    frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

    # YOLOv8 inference
    results = model(frame_bgr)[0]

    objects_detected_info = [] # Store tuples of (label, distance)
    current_time = time.time()

    # Loop through detections
    for det in results.boxes:
        label = model.names[int(det.cls[0])]

        # Only process if the detected object is a 'person'
        if label == 'person':
            # Calculate bounding box width in pixels
            x1, y1, x2, y2 = map(int, det.xyxy[0])
            pixel_width = x2 - x1

            distance = None # Initialize distance to None
            # Estimate distance
            if pixel_width > 0: # Avoid division by zero
                distance = distance_to_camera(KNOWN_WIDTH, FOCAL_LENGTH, pixel_width)


            # Check cooldown and add detection info
            if label not in last_spoken or current_time - last_spoken[label] > SPEAK_COOLDOWN:
                # Store label and distance for speaking later
                objects_detected_info.append((label, distance))
                last_spoken[label] = current_time # Update last spoken time for this label


            # Draw bounding box
            cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), (0, 255, 0), 2)
            # Add distance to the text displayed on the frame
            display_text = f"{label}: {distance:.2f} cm" if distance is not None else label
            cv2.putText(frame_bgr, display_text, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    # Generate speech audio based on collected information
    speech_text = ""
    audio_output = None

    if objects_detected_info:
        speech_phrases = []
        for label, distance in objects_detected_info:
            if label == 'person' and distance is not None:
                speech_phrases.append(f"person at {distance:.2f} centimeters")
            # You could add conditions for other objects here if needed in the future
            # elif label == 'some_other_object':
            #    speech_phrases.append(f"a {label}")

        speech_text = " , ".join(speech_phrases)

        if speech_text: # Only generate audio if there is text to speak
            # Save the speech to a temporary file
            temp_audio_file = "temp_speech.mp3"
            engine.save_to_file(speech_text, temp_audio_file)
            engine.runAndWait()
            # Read the audio file
            if os.path.exists(temp_audio_file):
                audio_output = temp_audio_file


    # Convert frame back to RGB for Gradio
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    # Return both the processed image and the audio file path
    return frame_rgb, audio_output

# Gradio Interface
iface = gr.Interface(
    fn=detect_objects,
    inputs=gr.Image(streaming=True),
    outputs=["image", gr.Audio(label="Detected Objects Audio", autoplay=True)],
    live=True,
    title="YOLOv8 Object Detection for Blind Assistance",
    description="Uses webcam feed to detect objects and speaks them aloud."
)

iface.launch()

0: 384x640 3 persons, 712.1ms
Speed: 3.2ms preprocess, 712.1ms inference, 180.7ms postprocess per image at shape (1, 3, 384, 640)
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()

0: 384x640 3 persons, 171.0ms
Speed: 4.5ms preprocess, 171.0ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)
Audio saved to temp_speech.mp3

0: 384x640 3 persons, 178.7ms
Speed: 2.5ms preprocess, 178.7ms inference, 2.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 156.8ms
Speed: 3.4ms preprocess, 156.8ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 155.9ms
Speed: 3.1ms preprocess, 155.9ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)
* Running on public URL: https

## Refine distance estimation (optional but recommended)

### Subtask:
Explain that for accurate distance estimation, calibration with known distances and object sizes is necessary and suggest steps for calibration.


**Reasoning**:
Add a markdown cell to explain the importance of calibration for accurate distance estimation, define the calibration variables, suggest a calibration method, and emphasize the importance of environmental calibration.



In [15]:
# Add a markdown cell with the following content:
"""
## Importance of Calibration for Distance Estimation

Accurate distance estimation using object detection and camera parameters relies heavily on proper calibration. The basic principle used here is similar triangles: if we know the real-world size of an object and its size in pixels in the image, and we know the camera's focal length, we can estimate the distance to the object.

However, the camera's focal length is a crucial parameter that needs to be determined for your specific camera and setup. The variables used in the code for distance estimation are:

- `KNOWN_DISTANCE`: This is the distance from the camera to an object of known size during the calibration process.
- `KNOWN_WIDTH`: This is the real-world width of the object used for calibration (e.g., the actual width of a person's shoulders or another easily measurable object).
- `FOCAL_LENGTH`: This represents the focal length of the camera. It's a measure of how strongly the camera converges light. In our context, it's a value that relates the real-world size of an object, its distance, and its size in pixels.

These variables are essential for the `distance_to_camera` function, which uses the formula:

`Distance = (KNOWN_WIDTH * FOCAL_LENGTH) / Pixel_Width`

Without accurate values for `KNOWN_WIDTH` and `FOCAL_LENGTH`, the distance estimations will not be reliable.

### Calibration Method

A simple method to calibrate the `FOCAL_LENGTH` is as follows:

1.  Place an object of a known width (`KNOWN_WIDTH`) at a precisely measured known distance (`KNOWN_DISTANCE`) from your camera.
2.  Capture an image of this object using the camera you will be using for object detection.
3.  Use an image editing tool or simple code to measure the width of the object in pixels (`Pixel_Width`) in the captured image.
4.  Calculate the focal length using the formula derived from the distance formula:

    `FOCAL_LENGTH = (Pixel_Width * KNOWN_DISTANCE) / KNOWN_WIDTH`

Once you have calculated the focal length, you can update the `FOCAL_LENGTH` variable in the code with this calibrated value.

### Environmental Considerations

For the most accurate results, it is recommended to perform this calibration in the environment where the object detection system will be used. Factors like lighting conditions, camera angle, and lens distortion can affect the pixel width of objects and thus the distance estimation. Calibrating in the target environment helps to account for these real-world variables.

Remember that the `KNOWN_WIDTH` for a 'person' is an average and can vary. For more precise distance estimation of people, you might consider using a more robust method that accounts for variations in human size or using a different known object for calibration.
"""

"\n## Importance of Calibration for Distance Estimation\n\nAccurate distance estimation using object detection and camera parameters relies heavily on proper calibration. The basic principle used here is similar triangles: if we know the real-world size of an object and its size in pixels in the image, and we know the camera's focal length, we can estimate the distance to the object.\n\nHowever, the camera's focal length is a crucial parameter that needs to be determined for your specific camera and setup. The variables used in the code for distance estimation are:\n\n- `KNOWN_DISTANCE`: This is the distance from the camera to an object of known size during the calibration process.\n- `KNOWN_WIDTH`: This is the real-world width of the object used for calibration (e.g., the actual width of a person's shoulders or another easily measurable object).\n- `FOCAL_LENGTH`: This represents the focal length of the camera. It's a measure of how strongly the camera converges light. In our context,

## Summary:

### Data Analysis Key Findings

*   The code was successfully modified to filter object detections, focusing solely on 'person' objects.
*   A distance estimation function was implemented using the principle of triangle similarity, incorporating placeholder values for `KNOWN_DISTANCE`, `KNOWN_WIDTH`, and `FOCAL_LENGTH`.
*   The bounding box for detected persons now displays the estimated distance in centimeters.
*   The speech output was updated to announce "person at \[estimated distance] centimeters" for detected persons.
*   An explanation of the importance of calibration for accurate distance estimation, including a simple calibration method, was prepared.

### Insights or Next Steps

*   The current distance estimations rely on placeholder calibration values. The next crucial step is to perform a proper calibration using a known object at a known distance to determine the accurate `FOCAL_LENGTH` for the camera setup.
*   While the system speaks the distance of a person, consider refining the speech output to handle scenarios with multiple detected people, perhaps indicating the closest person or listing distances for several individuals.
